<a href="https://colab.research.google.com/github/Abimanyu-07/english-to-hinglish_Translation/blob/main/Translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# English to Hinglish Translation



In [97]:
import sys
import torch
print("GPU available:", torch.cuda.is_available())

GPU available: True


In [98]:
import warnings
import numpy as np
import pandas as pd
import os
# from sacrebleu.metrics import BLEU
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
import pandas as pd
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yash9439/eng-hinglish-machine-translation")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'eng-hinglish-machine-translation' dataset.
Path to dataset files: /kaggle/input/eng-hinglish-machine-translation


In [99]:
# Define separate variables for each file path
dataset_dir = os.path.join(path, "Eng - Hinglish")

train_path = os.path.join(dataset_dir, "train.txt")
dev_path = os.path.join(dataset_dir, "dev.txt")
test_path = os.path.join(dataset_dir, "test.txt")

print("Train path:", train_path)
print("Dev path:", dev_path)
print("Test path:", test_path)

Train path: /kaggle/input/eng-hinglish-machine-translation/Eng - Hinglish/train.txt
Dev path: /kaggle/input/eng-hinglish-machine-translation/Eng - Hinglish/dev.txt
Test path: /kaggle/input/eng-hinglish-machine-translation/Eng - Hinglish/test.txt


In [100]:
def read_file(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.read()
    return lines

In [101]:
test_content = read_file(test_path)
train_content = read_file(train_path)
dev_content = read_file(dev_path)

test_array = test_content.strip().split("\n")
train_array = train_content.strip().split("\n")
dev_array = dev_content.strip().split("\n")

In [102]:
# Split each line by tab to separate input (English) and output (Hinglish)
test_split = [line.split("\t") for line in test_array if "\t" in line]
train_split = [line.split("\t") for line in train_array if "\t" in line]
dev_split = [line.split("\t") for line in dev_array if "\t" in line]

In [103]:
df = {}
test = pd.DataFrame(test_split, columns=["input", "output"])
train = pd.DataFrame(train_split, columns=["input", "output"])
dev = pd.DataFrame(dev_split, columns=["input", "output"])

from datasets import Dataset, DatasetDict

df["test"] = Dataset.from_pandas(test)
df["train"] = Dataset.from_pandas(train)
df["dev"] = Dataset.from_pandas(dev)

dataset = DatasetDict(df)

In [104]:
# Display the corrected test dataset DataFrame and Hugging Face Dataset format
print("Test DataFrame Preview:")
display(test.head())

print("\nHF Dataset Dictionary:")
display(dataset)

Test DataFrame Preview:


,input,output
0,hello,hello
1,"hello there , I have not seen this movie so im...","हैलो यार , मैं इस movie को नहीं देख हों तो , त..."
2,Alright that is fine . What is the movie ?,अच्छा तो इस movie किस बारे में हैं ?
3,The movie is The Social Network,is movie tho social network के बारे में हैं
4,I have not seen that one either .,मैं ऐसे कुछ नहीं देखा हूँ



HF Dataset Dictionary:


DatasetDict({
    test: Dataset({
        features: ['input', 'output'],
        num_rows: 942
    })
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 7040
    })
    dev: Dataset({
        features: ['input', 'output'],
        num_rows: 1020
    })
})

In [105]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [106]:
eng_sentences = train["input"].astype(str).tolist()
hin_sentences = train["output"].astype(str).tolist()

# Add <SOS> and <EOS> tokens to Hindi (target)
hin_sentences = [f"<SOS> {s} <EOS>" for s in hin_sentences]

In [107]:
hin_sentences[:10]

['<SOS> batman vs superman  <EOS>',
 '<SOS> Zack Snyder director हैं , 27 % Rotten Tomatoes , 4.9 / 10 .  <EOS>',
 '<SOS> लगता है बहुत popular नहीं है  <EOS>',
 '<SOS> but audience ने like किया , इसका cinema score भी है  <EOS>',
 '<SOS> yes  <EOS>',
 '<SOS> huge divergence है profession critic की opinion और रुगलर movie देखने वालों के बीच  <EOS>',
 '<SOS> मैंने तो कभी नहीं देखा है .  <EOS>',
 '<SOS> मुझे difference पाता है  <EOS>',
 '<SOS> मुझे तो believe ही नहीं होता कि उन्होंने Ben Affleck को Bataman बनाया .  <EOS>',
 '<SOS> So Probably इसको कोई award नहीं मिला , but box office पर इसको बधिया करना था  <EOS>']

In [108]:
from collections import Counter
import re

def tokenize(text):
    return text.strip().split()

# Build vocab
def build_vocab(sentences):
    tokens = []
    for s in sentences:
        tokens.extend(tokenize(s))
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, _ in Counter(tokens).items():
        vocab[word] = len(vocab)
    return vocab

src_vocab = build_vocab(eng_sentences)
tgt_vocab = build_vocab(hin_sentences)

# Reverse vocab for decoding
inv_tgt_vocab = {i: w for w, i in tgt_vocab.items()}

src_idx2word = {v: k for k, v in src_vocab.items()}
tgt_idx2word = {v: k for k, v in tgt_vocab.items()}

In [109]:
# encode
def encode_sentence(sentence, vocab, max_len):
    tokens = tokenize(sentence)
    # Map tokens to their indices, using <UNK> (1) for out-of-vocabulary words
    indices = [vocab.get(word, 1) for word in tokens]

    # Truncate if longer than max_len
    if len(indices) > max_len:
        indices = indices[:max_len]
    else:
        # Pad with <PAD> (0) up to max_len
        indices = indices + [0] * (max_len - len(indices))

    return torch.tensor(indices, dtype=torch.long)

MAX_LEN_SRC = max(len(tokenize(s)) for s in eng_sentences)
MAX_LEN_TGT = max(len(tokenize(s)) for s in hin_sentences)

src_encoded = [encode_sentence(s, src_vocab, MAX_LEN_SRC) for s in eng_sentences]
tgt_encoded = [encode_sentence(s, tgt_vocab, MAX_LEN_TGT) for s in hin_sentences]

In [110]:
print(len(src_encoded[1]))

283


In [111]:
print(len(tgt_vocab))
print(len(src_vocab))

8670
6519


In [112]:
# dataLoader
class TranslationDataset(Dataset):
    def __init__(self, df, src_vocab, tgt_vocab):
        self.src_sentences = df["input"].tolist()
        self.tgt_sentences = df["output"].tolist()
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab

    def __len__(self):
        return len(self.src_sentences)

    def __getitem__(self, idx):
        src_tensor = encode_sentence(self.src_sentences[idx], self.src_vocab)
        tgt_tensor = encode_sentence(self.tgt_sentences[idx], self.tgt_vocab)
        return src_tensor, tgt_tensor

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    src_batch = pad_sequence(src_batch, padding_value=0, batch_first=True)
    tgt_batch = pad_sequence(tgt_batch, padding_value=0, batch_first=True)
    return src_batch, tgt_batch

dataset = TranslationDataset(train, src_vocab, tgt_vocab)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)


In [113]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell

In [114]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden, cell):
        x = x.unsqueeze(1)  # (batch, 1)
        embedded = self.embedding(x)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc(output.squeeze(1))
        return prediction, hidden, cell

In [115]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        tgt_len = tgt.size(1)
        tgt_vocab_size = len(tgt_vocab)

        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)

        x = tgt[:, 0]  # <sos>
        for t in range(1, tgt_len):
            output, hidden, cell = self.decoder(x, hidden, cell)
            outputs[:, t] = output
            best_guess = output.argmax(1)
            x = tgt[:, t] if torch.rand(1).item() < teacher_forcing_ratio else best_guess

        return outputs

In [121]:
input_size_encoder = len(src_vocab)
input_size_decoder = len(tgt_vocab)
embed_size = 128
hidden_size = 256
learning_rate = 0.001
epochs = 10

In [117]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = Encoder(input_size_encoder, embed_size, hidden_size).to(device)
decoder = Decoder(input_size_decoder, embed_size, hidden_size).to(device)
model = Seq2Seq(encoder, decoder, device).to(device)

In [118]:
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss(ignore_index=0)

In [123]:
train_subset = train.iloc[:500].reset_index(drop=True)

# 2. Redefine TranslationDataset with proper max lengths to resolve the TypeError
class TranslationDataset(Dataset):
    def __init__(self, df, src_vocab, tgt_vocab, max_len_src, max_len_tgt):
        self.src_sentences = df["input"].tolist()
        self.tgt_sentences = df["output"].tolist()
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_len_src = max_len_src
        self.max_len_tgt = max_len_tgt

    def __len__(self):
        return len(self.src_sentences)

    def __getitem__(self, idx):
        src_tensor = encode_sentence(self.src_sentences[idx], self.src_vocab, self.max_len_src)
        tgt_tensor = encode_sentence(self.tgt_sentences[idx], self.tgt_vocab, self.max_len_tgt)
        return src_tensor, tgt_tensor
dataset = TranslationDataset(train_subset, src_vocab, tgt_vocab, MAX_LEN_SRC, MAX_LEN_TGT)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)

# 4. Training loop
for epoch in range(epochs):
    loop = tqdm(dataloader, leave=False)
    total_loss = 0
    for src_batch, tgt_batch in loop:
        src_batch = src_batch.to(device)
        tgt_batch = tgt_batch.to(device)

        output = model(src_batch, tgt_batch)
        output = output[:, 1:].reshape(-1, output.shape[2])
        tgt_batch = tgt_batch[:, 1:].reshape(-1)

        loss = criterion(output, tgt_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_description(f"Epoch [{epoch+1}/{epochs}]")
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataloader):.4f}")

Epoch 1/10, Loss: nan


Epoch 2/10, Loss: 5.4043


Epoch 3/10, Loss: 5.0555


Epoch 4/10, Loss: nan


Epoch 5/10, Loss: nan


Epoch 6/10, Loss: nan


Epoch 7/10, Loss: nan


Epoch 8/10, Loss: nan


Epoch 9/10, Loss: 3.1241


Epoch 10/10, Loss: nan


In [124]:
def translate_sentence(sentence, src_vocab, tgt_vocab, model, device, max_length=50):
    model.eval()

    # Encode the input sentence into indices (passing the required MAX_LEN_SRC)
    src_tensor = encode_sentence(sentence, src_vocab, MAX_LEN_SRC).unsqueeze(0).to(device)  # shape: (1, max_len)

    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)

    # First input to decoder is <SOS>
    tgt_indices = [tgt_vocab["<SOS>"]]
    cur_input = torch.tensor([tgt_vocab["<SOS>"]], dtype=torch.long).to(device)

    for _ in range(max_length):
        with torch.no_grad():
            output, hidden, cell = model.decoder(cur_input, hidden, cell)
            best_guess = output.argmax(1).item()

        tgt_indices.append(best_guess)

        if best_guess == tgt_vocab["<EOS>"]:
            break

        cur_input = torch.tensor([best_guess], dtype=torch.long).to(device)

    # Convert indices back to words
    inv_tgt_vocab = {idx: word for word, idx in tgt_vocab.items()}
    translated_tokens = [inv_tgt_vocab.get(idx, "<UNK>") for idx in tgt_indices]

    # Remove <SOS> and <EOS>
    return [token for token in translated_tokens if token not in ["<SOS>", "<EOS>", "<PAD>"]]

sentence = "the movie is amazing i enjoyed very much"

In [125]:
translation = translate_sentence(sentence, src_vocab, tgt_vocab, model, device)
print("Translated:", " ".join(translation))

Translated: में में कौन सितारे हैं obviously . वह एक है है और वह impossible हैं हैं . . . है . . है . . है . . है . . है . . है . . है . . है . . है . . है . . है


In [126]:
import nltk
from nltk.translate.bleu_score import corpus_bleu
nltk.download('punkt', quiet=True)

def evaluate_bleu(df_test, src_vocab, tgt_vocab, model, device):
    references = []
    hypotheses = []

    # Evaluate on a subset of test data for speed
    eval_subset = df_test.iloc[:200]

    for idx, row in eval_subset.iterrows():
        src_sentence = row['input']
        tgt_sentence = row['output']

        # Get model translation
        translated_tokens = translate_sentence(src_sentence, src_vocab, tgt_vocab, model, device)

        # Target reference tokens
        ref_tokens = tgt_sentence.strip().split()

        references.append([ref_tokens])
        hypotheses.append(translated_tokens)

    bleu_score = corpus_bleu(references, hypotheses)
    print(f"BLEU-4 Score on 200 Test Samples: {bleu_score * 100:.2f}%")

evaluate_bleu(test, src_vocab, tgt_vocab, model, device)

BLEU-4 Score on 200 Test Samples: 0.00%


/usr/local/lib/python3.13/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.13/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


In [127]:
unseen_sentences = [
    "the movie is amazing i enjoyed very much",
    "who is the director of this film",
    "i love watching movies with my family",
    "do you think this is a good choice",
    "i have not seen that movie yet"
]

print("--- Model Translation Showcase ---\n")
for sentence in unseen_sentences:
    translation = translate_sentence(sentence, src_vocab, tgt_vocab, model, device)
    print(f"English: {sentence}")
    print(f"Hinglish: {' '.join(translation)}\n")

--- Model Translation Showcase ---

English: the movie is amazing i enjoyed very much
Hinglish: में में कौन सितारे हैं obviously . वह एक है है और वह impossible हैं हैं . . . है . . है . . है . . है . . है . . है . . है . . है . . है . . है . . है

English: who is the director of this film
Hinglish: में में कौन सितारे हैं obviously . वह एक है है और वह impossible हैं हैं . . . है . . है . . है . . है . . है . . है . . है . . है . . है . . है . . है

English: i love watching movies with my family
Hinglish: में में कौन सितारे हैं obviously . वह एक है है और वह impossible हैं हैं . . . है . . है . . है . . है . . है . . है . . है . . है . . है . . है . . है

English: do you think this is a good choice
Hinglish: में में कौन सितारे हैं obviously . वह एक है है और वह impossible हैं हैं . . . है . . है . . है . . है . . है . . है . . है . . है . . है . . है . . है

English: i have not seen that movie yet
Hinglish: में में कौन सितारे हैं obviously . वह एक है है और वह impossible हैं हैं . . . है . 